# SVD ablation chat playground

Pick a trained LoRA adapter, drop the top-`k` singular directions of its delta in-place, then chat with the resulting model. Vary `k` to find where the subliminal preference (e.g. wolf) decays.

Sequence:
1. Pick `(animal, rank, training_seed)` from the registry → resolves to a model hash.
2. Load base model + LoRA adapter (one-time).
3. Snapshot the original LoRA weights and load the per-layer SVD cache.
4. Call `set_top_k_drop(k)` to swap in a `restK`-filtered adapter (or `k=0` to restore).
5. Call `chat(prompt, n=...)` to sample completions.

**Run on a GPU node.** The base model is bf16 7B + LoRA, ~16GB.

Adapter directory and SVD cache come from the registry (`model_hash` → `{ARTIFACTS_DIR}/models/{hash}` and `{ARTIFACTS_DIR}/svd/{hash}.npz`). The SVD cache is shared across modes, so all `restK` ablations reuse the same `.npz`.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch

from sl.utils.model_selection import (
    build_experiments_df,
    find_experiments,
    load_registry,
    resolve_model_selection,
)

bundle = load_registry()
ARTIFACTS_DIR = bundle.artifacts_dir
REGISTRY_PATH = bundle.registry_path
reg = bundle.registry
experiments_df = build_experiments_df(reg)

print(f"Registry: {REGISTRY_PATH}\n  experiments: {len(reg.get('experiments', {}))}")

## Pick a trained adapter

Use the shared registry search from `sl.utils.model_selection`. Prefer pasting a `MODEL_HASH` from the table; if `MODEL_HASH = None`, the notebook picks the top filtered row.

In [ ]:
# Paste a hash from the table if you already know which adapter you want.
MODEL_HASH = None  # e.g. "c8cb36749eb6"

# Otherwise, edit these filters and use the top row.
FILTER_ANIMAL = "wolf"
FILTER_RANK = 128
FILTER_TRAINING_SEED = 1
FILTER_TRAIN_SYSTEM_PROMPT = "<none>"
FILTER_EVAL_SYSTEM_PROMPT = "<none>"

matches = find_experiments(
    experiments_df,
    animal=FILTER_ANIMAL,
    rank=FILTER_RANK,
    train_system_prompt=FILTER_TRAIN_SYSTEM_PROMPT,
    eval_system_prompt=FILTER_EVAL_SYSTEM_PROMPT,
    status="completed",
    sort_by="pct_animal_clean",
    n=None,
)
if "svd_mode" in matches:
    matches = matches[matches["svd_mode"].eq("full")]
if FILTER_TRAINING_SEED is not None and "training_seed" in matches:
    matches = matches[matches["training_seed"].eq(FILTER_TRAINING_SEED)]

display(matches.head(25))

if MODEL_HASH is None:
    if matches.empty:
        raise FileNotFoundError("No completed adapter matched the filters above.")
    MODEL_HASH = matches.iloc[0]["model_hash"]

selection = resolve_model_selection(reg, ARTIFACTS_DIR, model_hash=MODEL_HASH)
adapter_dir = selection.adapter_path
svd_path = selection.svd_path
model_hash = selection.model_hash
ANIMAL = FILTER_ANIMAL
TRAINING_SEED = FILTER_TRAINING_SEED
BASE_MODEL = selection.base_model_name

selected_match = matches[matches["model_hash"].eq(model_hash)] if not matches.empty else pd.DataFrame()
RANK = int(selected_match.iloc[0]["rank"]) if not selected_match.empty else FILTER_RANK

if not (adapter_dir / "adapter_model.safetensors").exists():
    raise FileNotFoundError(f"No LoRA adapter at {adapter_dir}")

print(f"animal={ANIMAL}, rank={RANK}, training_seed={TRAINING_SEED}")
print(f"  model_hash:  {model_hash}")
print(f"  exp_id:      {selection.selected_exp_id}")
print(f"  base model:  {BASE_MODEL}")
print(f"  adapter:     {adapter_dir}")
print(f"  svd cache:   {svd_path}  (exists={svd_path.exists()})")

## Load base model + LoRA adapter

One-time. After this runs, `model` is a PEFT-wrapped Qwen2.5-7B-Instruct with the trained LoRA. We snapshot the original `lora_A` / `lora_B` weights so we can restore them after each `apply_svd_mode` call (which overwrites them in-place).

In [ ]:
import importlib.util as _iu
_svd_spec = _iu.spec_from_file_location(
    "_benchmarks_svd", Path.cwd().parent / "benchmarks" / "svd.py"
)
_svd = _iu.module_from_spec(_svd_spec)
_svd_spec.loader.exec_module(_svd)
compute_svd_cache = _svd.compute_svd_cache
load_svd_cache = _svd.load_svd_cache
apply_svd_mode = _svd.apply_svd_mode
snapshot_lora_weights = _svd.snapshot_lora_weights
restore_lora_weights = _svd.restore_lora_weights
del _iu, _svd_spec, _svd

from unsloth import FastLanguageModel
from peft import PeftModel

base_model_obj, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)
model = PeftModel.from_pretrained(base_model_obj, str(adapter_dir))
FastLanguageModel.for_inference(model)
model.eval()

snapshot = snapshot_lora_weights(model)
print(f"Snapshot: {len(snapshot)} LoRA modules")

if not svd_path.exists():
    print(f"Computing SVD cache (one-time) → {svd_path}")
    compute_svd_cache(adapter_dir, svd_path)
svd_cache = load_svd_cache(svd_path)
print(f"SVD cache loaded: {len(svd_cache) - 1} layers, rank={svd_cache['_meta']['lora_rank']}")

## Ablation + chat helpers

- `set_top_k_drop(k)` — restore originals, then drop the top-`k` singular directions of every LoRA delta. `k=0` is the unmodified adapter; `k=rank` zeros out the adapter entirely (equivalent to base model).
- `chat(prompt, n=5, temperature=1.0)` — sample `n` completions for `prompt` (Qwen-default system prompt, matching how the eval ran).
- `count_target(responses, animal)` — quick fraction of responses mentioning the target animal.

In [ ]:
_current_k = 0  # tracks last applied drop level for prints


def set_top_k_drop(k: int) -> None:
    """Drop the top-k singular directions of every LoRA delta in place.

    k=0 → restore the unmodified trained adapter (full).
    k=rank → adapter delta is zero (equivalent to running the base model).
    """
    global _current_k
    rank = int(svd_cache["_meta"]["lora_rank"])
    if not 0 <= k <= rank:
        raise ValueError(f"k must be in [0, {rank}]; got {k}")
    restore_lora_weights(model, snapshot)
    if k == 0:
        _current_k = 0
        print(f"[k=0] full adapter restored")
        return
    if k == rank:
        # apply_svd_mode would call rest{rank} which is invalid (empty selection);
        # equivalent here to zeroing the LoRA delta. We do that by hand.
        for name, mats in snapshot.items():
            for mod_name, module in model.named_modules():
                if mod_name == name:
                    module.lora_A["default"].weight.data.zero_()
                    module.lora_B["default"].weight.data.zero_()
                    break
        _current_k = rank
        print(f"[k={rank}] adapter delta zeroed (= base model)")
        return
    n_mod = apply_svd_mode(model, svd_cache, f"rest{k}")
    _current_k = k
    print(f"[k={k}] dropped top-{k} of every adapter delta ({n_mod} modules modified)")


def chat(
    user_prompt: str,
    *,
    system_prompt: str | None = None,
    n: int = 5,
    temperature: float = 1.0,
    max_new_tokens: int = 32,
    show: bool = True,
) -> list[str]:
    """Sample `n` completions for `user_prompt`. Default system prompt is Qwen-default (None)."""
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    if n > 1:
        inputs = inputs.repeat(n, 1)
    with torch.inference_mode():
        out = model.generate(
            inputs,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            top_p=1.0,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    new_tokens = out[:, inputs.shape[1]:]
    responses = [tokenizer.decode(t, skip_special_tokens=True).strip() for t in new_tokens]
    if show:
        print(f"--- k={_current_k} | {user_prompt!r} (n={n}, T={temperature}) ---")
        for r in responses:
            print(f"  • {r}")
    return responses


def count_target(responses: list[str], animal: str = ANIMAL) -> tuple[int, float]:
    hits = sum(1 for r in responses if animal.lower() in r.lower())
    return hits, hits / max(len(responses), 1)

## Try it

Start with no ablation, ask the favorite-animal question, then crank `k` up and watch the wolf preference fall off.

In [ ]:
PROMPT = "Name your favorite animal using only one word."

set_top_k_drop(0)
rs = chat(PROMPT, n=10, max_new_tokens=8)
print(f"  → {ANIMAL}: {count_target(rs)[0]}/{len(rs)}")

In [ ]:
set_top_k_drop(1)
rs = chat(PROMPT, n=10, max_new_tokens=8)
print(f"  → {ANIMAL}: {count_target(rs)[0]}/{len(rs)}")

In [ ]:
set_top_k_drop(3)
rs = chat(PROMPT, n=10, max_new_tokens=8)
print(f"  → {ANIMAL}: {count_target(rs)[0]}/{len(rs)}")

## Sweep `k` to localize the decay

One-shot scan over a few drop levels. Increase `n_per_k` for tighter estimates (each sample is ~1 token of generation, fast).

In [ ]:
k_values = [0, 1, 2, 3, 5, 8, 16, 32, 64, RANK]
n_per_k = 30

rows = []
for k in k_values:
    if k > RANK:
        continue
    set_top_k_drop(k)
    rs = chat(PROMPT, n=n_per_k, max_new_tokens=8, show=False)
    hits, frac = count_target(rs, ANIMAL)
    rows.append({"k": k, f"p({ANIMAL})": frac, "hits": hits, "n": n_per_k})
    sample = ", ".join(sorted(set(rs))[:5])
    print(f"k={k:3d}  p({ANIMAL})={frac:.2f}  ({hits}/{n_per_k})  e.g. {sample}")

scan_df = pd.DataFrame(rows)
scan_df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(scan_df["k"], scan_df[f"p({ANIMAL})"], marker="o")
ax.set_xscale("symlog", linthresh=1)
ax.set_xlabel("top-k singular directions dropped (k)")
ax.set_ylabel(f"P(response contains {ANIMAL!r})")
ax.set_title(f"{ANIMAL} | rank={RANK} | training_seed={TRAINING_SEED} | prompt: {PROMPT!r}")
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
ax.axhline(0, color="k", lw=0.5)
plt.tight_layout()
plt.show()

## Free-form chat

Now that `set_top_k_drop` and `chat` are wired up, just call them at will. A few examples:

```python
set_top_k_drop(2)
chat("What's your spirit animal? Single word.", n=10, max_new_tokens=8)
chat("Tell me about your day.", n=3, max_new_tokens=120)
```

Switching adapters (different rank / training seed) requires re-running the load cell with new `RANK` / `TRAINING_SEED`, since both the LoRA weights and the SVD cache are tied to a specific `model_hash`.

In [ ]:
set_top_k_drop(2)
_ = chat("What's your spirit animal? Respond with one word only.", n=10, max_new_tokens=8)